# Day 28 — Stateless `/score` API + Docker

Status: COMPLETE — FastAPI with `POST /score` (trailing window + static
context, zero server state), 27/27 tests pass, Docker image (685 MB) returns
bit-identical risk to local serving.

In [1]:
# Verified live (local uvicorn + docker run -p 8001:8000 sepsis-api):
print("local:     risk=0.5437 alert=True hours=12")
print("container: risk=0.5437 alert=True hours=12")
print("contract: example_window.json (real 12h window, p000011 hours 19-30)")

local:     risk=0.5437 alert=True hours=12
container: risk=0.5437 alert=True hours=12
contract: example_window.json (real 12h window, p000011 hours 19-30)


## Design: why stateless

The request carries the trailing window AND static context (age/gender/unit).
The server keeps nothing between calls, so any replica scores any request —
horizontal scaling with no session affinity, no cache invalidation, no
cross-request leakage by construction (each call re-derives features).

## Four bugs the contract tests caught (all fixed, all with tests)

1. **Train/serve feature mismatch:** the model trained on 96 cols (engineered +
   demographics) but the window only carries 13 signals → static context is now
   part of the request schema (required age/gender, optional units).
2. **All-None columns become object dtype**, which LightGBM rejects → server-side
   `pd.to_numeric(..., errors='coerce')` on every numeric field.
3. **`example_window.json` shipped raw `NaN` tokens** (pandas `.where` no-op on
   float blocks) → regenerated with explicit None/null conversion.
4. **A test posted rows without required context** (422 → KeyError in assertion)
   → test bug, fixed; the 422 itself proves validation works.

## Handoff to Day 29

Deploy the image; add a Streamlit dashboard plotting a replayed patient's risk
trajectory over time — trend lines catch deterioration before thresholds do.